# Week 3: 逻辑回归 (Logistic Regression)

本notebook包含Week 3的所有练习内容：
1. 分类 vs 回归 (Classification vs Regression)
2. Sigmoid 函数
3. 逻辑回归 (Logistic Regression)
4. 逻辑损失 (Logistic Loss)
5. 代价函数 (Cost Function)
6. 梯度下降 (Gradient Descent)
7. Scikit-learn 逻辑回归
8. 正则化 (Regularization)

## 练习 1: 分类 vs 回归

理解分类问题与回归问题的区别

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_common import dlc, plot_data
from plt_one_addpt_onclick import plt_one_addpt_onclick

### 创建分类数据集

In [ ]:
# 一维分类数据
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0, 0, 0, 1, 1, 1])  # 0: 负类, 1: 正类

# 二维分类数据
X_train2 = np.array([[0.5, 1.5], [1, 1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_train2 = np.array([0, 0, 0, 1, 1, 1])

# 创建正类和负类的掩码
pos = y_train == 1
neg = y_train == 0
print(f"正类掩码: {pos}")
print(f"负类掩码: {neg}")

### 可视化分类数据

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# 一维数据
ax[0].scatter(x_train[pos], y_train[pos], color='red', marker='o', label='正类 (y=1)')
ax[0].scatter(x_train[neg], y_train[neg], color='blue', marker='x', label='负类 (y=0)')
ax[0].set_xlabel('X')
ax[0].set_ylabel('y')
ax[0].set_title('一维分类问题')
ax[0].legend(loc='upper left')

# 二维数据
plot_data(X_train2, y_train2, ax[1])
ax[1].axis([0, 5, 0, 5])
ax[1].set_xlabel('$x_0$')
ax[1].set_ylabel('$x_1$')
ax[1].set_title('二维分类问题')
ax[1].legend(loc='upper left')

plt.tight_layout()
plt.show()

### 交互式示例：线性回归用于分类的问题

In [ ]:
# 使用线性回归处理分类问题的交互式示例
w_in = np.zeros((1))
b_in = 0
plt.close('all')

# 点击图形添加数据点，观察线性回归的局限性
addpt = plt_one_addpt_onclick(x_train, y_train, w_in, b_in, logistic=False)
print("提示: 线性回归对分类问题不太适用，因为输出值可能超出[0,1]范围")

## 练习 2: Sigmoid 函数

学习Sigmoid函数及其在逻辑回归中的作用

In [ ]:
from lab_utils_common import draw_vthresh

### 理解指数函数

In [ ]:
# NumPy的exp函数
input_array = np.array([1, 2, 3])
exp_array = np.exp(input_array)
print(f"输入数组: {input_array}")
print(f"exp数组: {exp_array}")

# 单个值
input_val = 1
exp_val = np.exp(input_val)
print(f"\n输入值: {input_val}")
print(f"exp值: {exp_val}")

### 实现Sigmoid函数

Sigmoid函数: $\sigma(z) = \frac{1}{1 + e^{-z}}$

特性:
- 输出范围: (0, 1)
- $\sigma(0) = 0.5$
- 当z很大时，$\sigma(z) \approx 1$
- 当z很小时，$\sigma(z) \approx 0$

In [ ]:
def sigmoid(z):
    """
    计算sigmoid函数
    
    参数:
        z (ndarray): 输入值
    
    返回:
        g (ndarray): sigmoid(z)
    """
    return 1 / (1 + np.exp(-z))

### 可视化Sigmoid函数

In [ ]:
# 创建测试数据
z_tmp = np.arange(-10, 11)
y = sigmoid(z_tmp)

np.set_printoptions(precision=3)
print(f"z: {z_tmp}")
print(f"sigmoid(z): {y}")

# 绘制sigmoid曲线
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(z_tmp, y, c="b", linewidth=2)
ax.set_xlabel('z')
ax.set_ylabel('sigmoid(z)')
ax.set_title('Sigmoid函数')
ax.grid(True)
draw_vthresh(ax, 0)
plt.show()

### 交互式示例：逻辑回归

In [ ]:
# 使用逻辑回归处理分类问题
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0, 0, 0, 1, 1, 1])

w_in = np.zeros((1))
b_in = 0

plt.close('all')
# 点击图形添加数据点，观察逻辑回归如何工作
addpt = plt_one_addpt_onclick(x_train, y_train, w_in, b_in, logistic=True)
plt.show()

print("提示: 逻辑回归的输出始终在[0,1]范围内，更适合分类问题")

## 练习 3: 逻辑回归

理解逻辑回归模型和决策边界

In [ ]:
from lab_utils_common import plot_data, sigmoid, draw_vthresh

plt.style.use('./deeplearning.mplstyle')

### 二维分类数据

In [ ]:
X = np.array([[0.5, 1.5], [1, 1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y = np.array([0, 0, 0, 1, 1, 1]).reshape(-1, 1)

print("X shape:", X.shape, "\nX:\n", X)
print("y shape:", y.shape, "\ny:\n", y)

# 可视化数据
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
plot_data(X, y, ax)
ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$')
ax.set_xlabel('$x_0$')
ax.set_title('训练数据')
plt.show()

### 逻辑回归模型

模型: $f_{\mathbf{w},b}(\mathbf{x}) = g(\mathbf{w} \cdot \mathbf{x} + b)$

其中 $g(z) = \frac{1}{1+e^{-z}}$ 是sigmoid函数

In [ ]:
# 再次可视化sigmoid函数
z = np.arange(-10, 11)
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.plot(z, sigmoid(z), label='sigmoid', linewidth=2)
ax.set_title('Sigmoid函数')
ax.set_xlabel('z')
ax.set_ylabel('$\\sigma(z)$')
ax.grid(True)
draw_vthresh(ax, 0)
plt.show()

### 决策边界

决策边界是将特征空间分为两部分的线（或曲线）：
- $\mathbf{w} \cdot \mathbf{x} + b = 0$
- 当 $\mathbf{w} \cdot \mathbf{x} + b \geq 0$ 时，预测 $\hat{y} = 1$
- 当 $\mathbf{w} \cdot \mathbf{x} + b < 0$ 时，预测 $\hat{y} = 0$

In [ ]:
# 绘制决策边界示例
x0 = np.arange(0, 6)
x1 = 3 - x0  # 假设决策边界为 w0*x0 + w1*x1 + b = 0，即 x0 + x1 - 3 = 0

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.plot(x0, x1, c="b", linewidth=2, label='决策边界')
ax.axis([0, 4, 0, 3.5])

# 填充区域
ax.fill_between(x0, x1, alpha=0.2)

# 绘制数据点
plot_data(X, y, ax)
ax.set_ylabel(r'$x_1$')
ax.set_xlabel(r'$x_0$')
ax.set_title('逻辑回归的决策边界')
ax.legend()
plt.show()

print("蓝色区域: 预测为正类 (y=1)")
print("白色区域: 预测为负类 (y=0)")

## 练习 4: 逻辑损失 (Logistic Loss)

理解为什么不能对逻辑回归使用平方误差损失，以及逻辑损失函数的定义

In [ ]:
from plt_logistic_loss import plt_logistic_cost, plt_two_logistic_loss_curves
from plt_logistic_loss import soup_bowl, plt_simple_example, plt_logistic_squared_error

plt.style.use('./deeplearning.mplstyle')

### 回顾：线性回归的平方误差损失

线性回归使用平方误差，代价函数呈"碗"形

In [ ]:
# 线性回归的"碗"形代价函数
soup_bowl()
plt.close('all')

### 问题：逻辑回归使用平方误差

如果对逻辑回归使用平方误差，代价函数会有多个局部最小值

In [ ]:
x_train = np.array([0., 1, 2, 3, 4, 5], dtype=np.longdouble)
y_train = np.array([0, 0, 0, 1, 1, 1], dtype=np.longdouble)

# 简单示例
plt_simple_example(x_train, y_train)
plt.show()
plt.close('all')

# 平方误差代价函数的问题
plt_logistic_squared_error(x_train, y_train)
plt.show()

print("注意: 平方误差代价函数对逻辑回归是非凸的，有多个局部最小值!")

### 逻辑损失函数

单个样本的损失:

$$L(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) = \begin{cases}
-\log(f_{\mathbf{w},b}(\mathbf{x}^{(i)})) & \text{if } y^{(i)} = 1 \\
-\log(1 - f_{\mathbf{w},b}(\mathbf{x}^{(i)})) & \text{if } y^{(i)} = 0
\end{cases}$$

简化形式:
$$L(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) = -y^{(i)}\log(f_{\mathbf{w},b}(\mathbf{x}^{(i)})) - (1-y^{(i)})\log(1-f_{\mathbf{w},b}(\mathbf{x}^{(i)}))$$

In [ ]:
# 可视化两种情况的损失曲线
plt_two_logistic_loss_curves()

print("观察:")
print("- 当y=1时，如果预测接近1，损失很小；如果预测接近0，损失很大")
print("- 当y=0时，如果预测接近0，损失很小；如果预测接近1，损失很大")

### 逻辑回归的代价函数

整个数据集的代价:
$$J(\mathbf{w},b) = \frac{1}{m} \sum_{i=1}^{m} L(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)})$$

In [ ]:
plt.close('all')
cst = plt_logistic_cost(x_train, y_train)

print("逻辑损失函数是凸的，只有一个全局最小值!")

## 练习 5: 代价函数实现

实现逻辑回归的代价函数

In [ ]:
from lab_utils_common import plot_data, sigmoid, dlc

plt.style.use('./deeplearning.mplstyle')

### 准备数据

In [ ]:
X_train = np.array([[0.5, 1.5], [1, 1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_train = np.array([0, 0, 0, 1, 1, 1])

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
plot_data(X_train, y_train, ax)
ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.show()

### 实现代价函数

In [ ]:
def compute_cost_logistic(X, y, w, b):
    """
    计算逻辑回归的代价函数
    
    参数:
        X (ndarray): 特征矩阵 (m, n)
        y (ndarray): 目标值 (m,)
        w (ndarray): 权重 (n,)
        b (scalar): 偏置
    
    返回:
        cost (scalar): 代价
    """
    m = X.shape[0]
    cost = 0.0
    
    for i in range(m):
        z_i = np.dot(X[i], w) + b
        f_wb_i = sigmoid(z_i)
        cost += -y[i] * np.log(f_wb_i) - (1 - y[i]) * np.log(1 - f_wb_i)
    
    cost /= m
    return cost

### 测试代价函数

In [ ]:
# 测试参数
w_tmp = np.array([1, 1])
b_tmp = -3

cost = compute_cost_logistic(X_train, y_train, w_tmp, b_tmp)
print(f"代价 (b=-3): {cost:.4f}")

### 比较不同参数的代价

In [ ]:
# 可视化不同决策边界
x0 = np.arange(0, 6)
x1 = 3 - x0       # b = -3
x1_other = 4 - x0  # b = -4

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.plot(x0, x1, c=dlc['dlblue'], label='$b$=-3')
ax.plot(x0, x1_other, c=dlc['dlorange'], label='$b$=-4')
ax.axis([0, 4, 0, 4])

plot_data(X_train, y_train, ax)
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.legend()
plt.show()

# 计算两种情况的代价
w_array1 = np.array([1, 1])
b_1 = -3
w_array2 = np.array([1, 1])
b_2 = -4

cost1 = compute_cost_logistic(X_train, y_train, w_array1, b_1)
cost2 = compute_cost_logistic(X_train, y_train, w_array2, b_2)

print(f'\n代价 (b=-3): {cost1:.2f}')
print(f'代价 (b=-4): {cost2:.2f}')
print(f'\nb=-3的代价更低，说明这是更好的决策边界!')

## 练习 6: 梯度下降

实现逻辑回归的梯度下降算法

In [ ]:
import copy
import math
from lab_utils_common import dlc, plot_data, plt_tumor_data, sigmoid, compute_cost_logistic
from plt_quad_logistic import plt_quad_logistic, plt_prob

plt.style.use('./deeplearning.mplstyle')

### 准备数据

In [ ]:
X_train = np.array([[0.5, 1.5], [1, 1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_train = np.array([0, 0, 0, 1, 1, 1])

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
plot_data(X_train, y_train, ax)
ax.axis([0, 4, 0, 3])
ax.set_ylabel('$x_1$')
ax.set_xlabel('$x_0$')
plt.show()

### 实现梯度计算

梯度公式:
$$\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_j^{(i)}$$
$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})$$

In [ ]:
def compute_gradient_logistic(X, y, w, b):
    """
    计算逻辑回归的梯度
    
    参数:
        X (ndarray): 特征矩阵 (m, n)
        y (ndarray): 目标值 (m,)
        w (ndarray): 权重 (n,)
        b (scalar): 偏置
    
    返回:
        dj_dw (ndarray): 对w的梯度 (n,)
        dj_db (scalar): 对b的梯度
    """
    m, n = X.shape
    dj_dw = np.zeros((n,))
    dj_db = 0
    
    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i], w) + b)
        err_i = f_wb_i - y[i]
        for j in range(n):
            dj_dw[j] += err_i * X[i, j]
        dj_db += err_i
    
    dj_dw /= m
    dj_db /= m
    
    return dj_dw, dj_db

### 测试梯度计算

In [ ]:
X_tmp = np.array([[0.5, 1.5], [1, 1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_tmp = np.array([0, 0, 0, 1, 1, 1])
w_tmp = np.array([2., 3.])
b_tmp = 1

dj_dw_tmp, dj_db_tmp = compute_gradient_logistic(X_tmp, y_tmp, w_tmp, b_tmp)
print(f"dj_db: {dj_db_tmp:.4f}")
print(f"dj_dw: {dj_dw_tmp.tolist()}")

### 实现梯度下降

In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters):
    """
    执行梯度下降优化
    
    参数:
        X (ndarray): 特征矩阵
        y (ndarray): 目标值
        w_in (ndarray): 初始权重
        b_in (scalar): 初始偏置
        alpha (float): 学习率
        num_iters (int): 迭代次数
    
    返回:
        w (ndarray): 优化后的权重
        b (scalar): 优化后的偏置
        J_history (list): 代价历史
    """
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in
    
    for i in range(num_iters):
        # 计算梯度
        dj_dw, dj_db = compute_gradient_logistic(X, y, w, b)
        
        # 更新参数
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        
        # 保存代价
        if i < 10000:
            J_history.append(compute_cost_logistic(X, y, w, b))
        
        # 打印进度
        if i % math.ceil(num_iters / 10) == 0:
            print(f"迭代 {i:4d}, 代价: {J_history[-1]:.4f}")
    
    return w, b, J_history

### 运行梯度下降

In [ ]:
# 初始化参数
w_tmp = np.zeros_like(X_train[0])
b_tmp = 0
alpha = 0.1
num_iters = 10000

# 运行梯度下降
w, b, J_history = gradient_descent(X_train, y_train, w_tmp, b_tmp, alpha, num_iters)

print(f"\n最优参数: w = {w}, b = {b:.4f}")

### 可视化结果

In [ ]:
# 绘制决策边界和预测概率
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
plt_prob(ax, w, b)
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
ax.axis([0, 4, 0, 3])
plot_data(X_train, y_train, ax)

# 绘制决策边界线
x0 = -b / w[0]
x1 = -b / w[1]
ax.plot([0, x0], [x1, 0], c=dlc["dlblue"], lw=2, label='决策边界')
ax.legend()
plt.show()

print("背景颜色表示预测为正类的概率")

### 一维示例

In [ ]:
# 一维肿瘤数据示例
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0, 0, 0, 1, 1, 1])

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plt_tumor_data(x_train, y_train, ax)
plt.show()

In [ ]:
# 可视化代价函数
w_range = np.array([-1, 7])
b_range = np.array([1, -11])
quad = plt_quad_logistic(x_train, y_train, w_range, b_range)
plt.show()

print("代价函数是凸的，便于优化!")

## 练习 7: Scikit-learn 逻辑回归

使用Scikit-learn的LogisticRegression

In [ ]:
from sklearn.linear_model import LogisticRegression

### 训练逻辑回归模型

In [ ]:
# 数据
X = np.array([[0.5, 1.5], [1, 1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y = np.array([0, 0, 0, 1, 1, 1])

# 创建和训练模型
lr_model = LogisticRegression()
lr_model.fit(X, y)

print("模型训练完成!")

### 进行预测

In [ ]:
# 预测类别
y_pred = lr_model.predict(X)
print("预测类别:", y_pred)
print("真实类别:", y)

# 计算准确率
accuracy = lr_model.score(X, y)
print(f"\n训练集准确率: {accuracy * 100:.2f}%")

### 预测概率

In [ ]:
# 预测概率
y_prob = lr_model.predict_proba(X)
print("\n预测概率 [P(y=0), P(y=1)]:")
for i in range(len(X)):
    print(f"样本 {i}: {y_prob[i]}, 预测类别: {y_pred[i]}, 真实类别: {y[i]}")

### 查看模型参数

In [ ]:
# 获取参数
w_sklearn = lr_model.coef_[0]
b_sklearn = lr_model.intercept_[0]

print(f"\nScikit-learn模型参数:")
print(f"w = {w_sklearn}")
print(f"b = {b_sklearn:.4f}")

print(f"\n我们实现的模型参数:")
print(f"w = {w}")
print(f"b = {b:.4f}")

print("\n注意: 由于正则化等因素，参数可能略有不同")

## 练习 8: 正则化 (Regularization)

理解正则化如何防止过拟合

In [ ]:
from plt_overfit import overfit_example
from lab_utils_common import sigmoid

np.set_printoptions(precision=8)

### 正则化的线性回归代价函数

$$J(\mathbf{w},b) = \frac{1}{2m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2 + \frac{\lambda}{2m} \sum_{j=1}^{n} w_j^2$$

In [ ]:
def compute_cost_linear_reg(X, y, w, b, lambda_=1):
    """
    计算正则化的线性回归代价函数
    
    参数:
        X (ndarray): 特征矩阵 (m, n)
        y (ndarray): 目标值 (m,)
        w (ndarray): 权重 (n,)
        b (scalar): 偏置
        lambda_ (float): 正则化参数
    
    返回:
        total_cost (scalar): 总代价
    """
    m = X.shape[0]
    n = len(w)
    
    # 计算基本代价
    cost = 0.
    for i in range(m):
        f_wb_i = np.dot(X[i], w) + b
        cost += (f_wb_i - y[i]) ** 2
    cost /= (2 * m)
    
    # 计算正则化项
    reg_cost = 0
    for j in range(n):
        reg_cost += w[j] ** 2
    reg_cost = (lambda_ / (2 * m)) * reg_cost
    
    return cost + reg_cost

### 测试正则化线性回归代价

In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5, 6)
y_tmp = np.array([0, 1, 0, 1, 0])
w_tmp = np.random.rand(X_tmp.shape[1]) - 0.5
b_tmp = 0.5
lambda_tmp = 0.7

cost_tmp = compute_cost_linear_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)
print(f'正则化代价 (lambda={lambda_tmp:.2f}): {cost_tmp:.4f}')

### 正则化的逻辑回归代价函数

$$J(\mathbf{w},b) = \frac{1}{m} \sum_{i=1}^{m} L(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) + \frac{\lambda}{2m} \sum_{j=1}^{n} w_j^2$$

In [ ]:
def compute_cost_logistic_reg(X, y, w, b, lambda_=1):
    """
    计算正则化的逻辑回归代价函数
    
    参数:
        X (ndarray): 特征矩阵 (m, n)
        y (ndarray): 目标值 (m,)
        w (ndarray): 权重 (n,)
        b (scalar): 偏置
        lambda_ (float): 正则化参数
    
    返回:
        total_cost (scalar): 总代价
    """
    m, n = X.shape
    
    # 计算基本代价
    cost = 0.0
    for i in range(m):
        z_i = np.dot(X[i], w) + b
        f_wb_i = sigmoid(z_i)
        cost += -y[i] * np.log(f_wb_i) - (1 - y[i]) * np.log(1 - f_wb_i)
    cost /= m
    
    # 计算正则化项
    reg_cost = 0
    for j in range(n):
        reg_cost += w[j] ** 2
    reg_cost = (lambda_ / (2 * m)) * reg_cost
    
    return cost + reg_cost

### 测试正则化逻辑回归代价

In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5, 6)
y_tmp = np.array([0, 1, 0, 1, 0])
w_tmp = np.random.rand(X_tmp.shape[1]) - 0.5
b_tmp = 0.5
lambda_tmp = 0.7

cost_tmp = compute_cost_logistic_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)
print(f'正则化代价 (lambda={lambda_tmp:.2f}): {cost_tmp:.4f}')

### 正则化的线性回归梯度

$$\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_j^{(i)} + \frac{\lambda}{m}w_j$$
$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})$$

In [ ]:
def compute_gradient_linear_reg(X, y, w, b, lambda_):
    """
    计算正则化线性回归的梯度
    
    参数:
        X (ndarray): 特征矩阵 (m, n)
        y (ndarray): 目标值 (m,)
        w (ndarray): 权重 (n,)
        b (scalar): 偏置
        lambda_ (float): 正则化参数
    
    返回:
        dj_dw (ndarray): 对w的梯度 (n,)
        dj_db (scalar): 对b的梯度
    """
    m, n = X.shape
    dj_dw = np.zeros((n,))
    dj_db = 0.0
    
    for i in range(m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] += err * X[i][j]
        dj_db += err
    dj_dw /= m
    dj_db /= m
    
    # 添加正则化项
    for j in range(n):
        dj_dw[j] += (lambda_ / m) * w[j]
    
    return dj_dw, dj_db

### 测试正则化线性回归梯度

In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5, 3)
y_tmp = np.array([0, 1, 0, 1, 0])
w_tmp = np.random.rand(X_tmp.shape[1])
b_tmp = 0.5
lambda_tmp = 0.7

dj_dw_tmp, dj_db_tmp = compute_gradient_linear_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)
print(f'dJ/dw: {dj_dw_tmp}')
print(f'dJ/db: {dj_db_tmp:.8f}')

### 正则化的逻辑回归梯度

In [ ]:
def compute_gradient_logistic_reg(X, y, w, b, lambda_=1):
    """
    计算正则化逻辑回归的梯度
    
    参数:
        X (ndarray): 特征矩阵 (m, n)
        y (ndarray): 目标值 (m,)
        w (ndarray): 权重 (n,)
        b (scalar): 偏置
        lambda_ (float): 正则化参数
    
    返回:
        dj_dw (ndarray): 对w的梯度 (n,)
        dj_db (scalar): 对b的梯度
    """
    m, n = X.shape
    dj_dw = np.zeros((n,))
    dj_db = 0.0
    
    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i], w) + b)
        err = f_wb_i - y[i]
        for j in range(n):
            dj_dw[j] += err * X[i][j]
        dj_db += err
    dj_dw /= m
    dj_db /= m
    
    # 添加正则化项
    for j in range(n):
        dj_dw[j] += (lambda_ / m) * w[j]
    
    return dj_dw, dj_db

### 测试正则化逻辑回归梯度

In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5, 3)
y_tmp = np.array([0, 1, 0, 1, 0])
w_tmp = np.random.rand(X_tmp.shape[1])
b_tmp = 0.5
lambda_tmp = 0.7

dj_dw_tmp, dj_db_tmp = compute_gradient_logistic_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)
print(f'dj_dw: {dj_dw_tmp.tolist()}')
print(f'dj_db: {dj_db_tmp:.8f}')

## 总结

在本周的练习中，我们学习了:
1. **分类问题** - 理解分类与回归的区别
2. **Sigmoid函数** - 将线性输出映射到[0,1]范围
3. **逻辑回归** - 用于二分类问题的模型
4. **逻辑损失** - 为什么不能使用平方误差
5. **代价函数** - 逻辑回归的代价函数实现
6. **梯度下降** - 优化逻辑回归参数
7. **Scikit-learn** - 使用现成的逻辑回归实现
8. **正则化** - 防止过拟合的技术

逻辑回归是分类问题的基础算法，也是深度学习的基石！